# CGM pattern mining and hypoglycemia prediction

## Student Data Analysis Notebook

**Course:** I 320D - Data Science for Biomedical Informatics
**Instructor:** Ammar Darkazanli
**Semester:** Spring 2026

---

### Notebook Sections
| Part | Topic | Strategy |
|------|-------|----------|
| 1 | Setup and Load Data | — |
| 2 | Schema Harmonization | — |
| 3 | Vertical Stack | `pd.concat` |
| 4 | Aggregate Enrichment | `groupby` + `merge` |
| 5 | Train-Transfer | Model as Bridge |
| 6 | Challenge Exercises | All strategies |

---

### 📋 Dataset Overview

| | OhioT1DM |
|---|---|
| **Source** | The OhioT1DM dataset for blood glucose level prediction: Update 2020 |
| **Rows** | ? |
| **Columns** | ? |
| **Target** | 'CGM' |
| **Encoding style** | Strings: ?, Integers: ? |
| **Missing values** | ? |
| **Key additions** | ? |

---
# PART 1: Setup and Load Data
---

### 1.1 Import Libraries

In [27]:
# In case you need the installs

#!pip install pandas --quiet
#!pip install numpy --quiet
#!pip install matplotlib --quiet
#!pip install seaborn --quiet
#!pip install scipy --quiet
#!pip install scikit-learn --quiet
#pip install lxml

In [28]:
# TODO: Import the required libraries
# - pandas as pd
# - numpy as np
# - matplotlib.pyplot as plt
# - seaborn as sns
# - from sklearn.linear_model import LogisticRegression
# - from sklearn.preprocessing import StandardScaler

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import xml.etree.ElementTree as ET
from pathlib import Path
import os


# Display settings (run this after importing)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')

import warnings
warnings.filterwarnings('ignore')

print("Libraries Imported!")

Libraries Imported!


### 1.2 Load the OhioT1DM Dataset

In [29]:
# Convert all OhioT1DM XML files to CSV
# This creates separate CSV files for each event type (cgm, bolus, meal, etc.)

# Define directories
source_dir = Path(r"C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project\OhioT1DM\2020\train")
output_dir = Path(r"C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project")

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)

# Define event types and their attributes
event_types = {
    'cgm': ['ts', 'value'],
    'finger': ['ts', 'value'],
    'basal': ['ts', 'value', 'duration'],
    'bolus': ['ts', 'dose', 'bwz_carb_input'],
    'meal': ['ts', 'carbs'],
    'sleep': ['ts', 'quality'],
    'exercise': ['ts', 'intensity', 'duration'],
    'heartrate': ['ts', 'value'],
    'steps': ['ts', 'value'],
}

# Mapping of XML tags to event type names
tag_mapping = {
    'glucose_level': 'cgm',
    'finger_stick': 'finger',
    'basal': 'basal',
    'bolus': 'bolus',
    'meal': 'meal',
    'sleep': 'sleep',
    'exercise': 'exercise',
    'basis_heart_rate': 'heartrate',
    'basis_steps': 'steps',
}

# Initialize data storage
all_data = {key: [] for key in event_types.keys()}

# Get all XML files
xml_files = sorted(source_dir.glob('*.xml'))
print(f"Found {len(xml_files)} patient files")
print(f"Output directory: {output_dir}\n")

# Process each XML file
for i, xml_file in enumerate(xml_files, 1):
    try:
        tree = ET.parse(str(xml_file))
        root = tree.getroot()
        patient_id = xml_file.stem  # filename without extension
        
        # Extract events for each type
        for xml_tag, event_type in tag_mapping.items():
            for parent in root.iter(xml_tag):
                for event in parent.iter('event'):
                    row = {'patient_id': patient_id}
                    for attr in event_types[event_type]:
                        row[attr] = event.get(attr)
                    all_data[event_type].append(row)
        
        if i % 2 == 0:
            print(f"  Processed {i}/{len(xml_files)} files...")
    
    except Exception as e:
        print(f"  Error parsing {xml_file.name}: {e}")

print(f"\nProcessed all {len(xml_files)} files\n")

# Convert to DataFrames and save as CSV
for event_type, records in all_data.items():
    if records:
        df = pd.DataFrame(records)
        
        # Parse timestamps and numeric values
        df['ts'] = pd.to_datetime(df['ts'], format='%d-%m-%Y %H:%M:%S', errors='coerce')
        for col in df.columns:
            if col not in ['patient_id', 'ts']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Sort by timestamp
        df = df.sort_values('ts').reset_index(drop=True)
        
        # Save to CSV
        output_file = output_dir / f"{event_type}.csv"
        df.to_csv(output_file, index=False)
        print(f"✓ {event_type}.csv: {len(df):,} records from {df['patient_id'].nunique()} patients")
    else:
        print(f"✗ {event_type}.csv: No data")

print(f"\n✅ Conversion complete!")
print(f"CSV files saved to:\n{output_dir}")

Found 6 patient files
Output directory: C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project

  Processed 2/6 files...
  Processed 4/6 files...
  Processed 6/6 files...

Processed all 6 files

✓ cgm.csv: 65,535 records from 6 patients
✓ finger.csv: 1,691 records from 6 patients
✓ basal.csv: 357 records from 6 patients
✓ bolus.csv: 1,568 records from 6 patients
✓ meal.csv: 702 records from 6 patients
✓ sleep.csv: 150 records from 5 patients
✓ exercise.csv: 61 records from 4 patients
✗ heartrate.csv: No data
✗ steps.csv: No data

✅ Conversion complete!
CSV files saved to:
C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project


In [30]:
def load_ohiot1dm_csv(csv_dir):
    """
    Load OhioT1DM data from pre-converted CSV files.
    
    Parameters:
    -----------
    csv_dir : str
        Path to directory containing CSV files (cgm.csv, bolus.csv, etc.)
    
    Returns:
    --------
    dict : Dictionary with keys for each event type
    """
    
    event_types = ['cgm', 'finger', 'basal', 'bolus', 'meal', 'sleep', 'exercise', 'heartrate', 'steps']
    data = {}
    
    csv_path = Path(csv_dir)
    print(f"Loading CSV files from: {csv_path}\n")
    
    for event_type in event_types:
        csv_file = csv_path / f"{event_type}.csv"
        
        if csv_file.exists():
            try:
                df = pd.read_csv(csv_file)
                
                # Parse timestamp
                if 'ts' in df.columns:
                    df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
                
                # Convert numeric columns
                for col in df.columns:
                    if col not in ['patient_id', 'ts']:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                
                # Sort by timestamp
                df = df.sort_values('ts').reset_index(drop=True)
                
                data[event_type] = df
                print(f"✓ {event_type}.csv: {len(df):,} records from {df['patient_id'].nunique()} patients")
            
            except Exception as e:
                print(f"✗ {event_type}.csv: Error loading - {e}")
                data[event_type] = pd.DataFrame()
        else:
            print(f"✗ {event_type}.csv: File not found")
            data[event_type] = pd.DataFrame()
    
    return data

# Load the CSV data
csv_dir = r"C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project"
data = load_ohiot1dm_csv(csv_dir)

cgm = data['cgm']
print(f"\n{'='*60}")
print(f"CGM Data Loaded: {cgm.shape[0]:,} glucose readings")
print(f"{'='*60}")
print("\nFirst 5 rows:")
print(cgm.head())

Loading CSV files from: C:\Users\tyler\data_science_projects\CGM_hypoglycemia_prediction_project

✓ cgm.csv: 65,535 records from 6 patients
✓ finger.csv: 1,691 records from 6 patients
✓ basal.csv: 357 records from 6 patients
✓ bolus.csv: 1,568 records from 6 patients
✓ meal.csv: 702 records from 6 patients
✓ sleep.csv: 150 records from 5 patients
✓ exercise.csv: 61 records from 4 patients
✗ heartrate.csv: File not found
✗ steps.csv: File not found

CGM Data Loaded: 65,535 glucose readings

First 5 rows:
        patient_id                  ts  value
0  552-ws-training 2025-04-16 11:17:05     95
1  552-ws-training 2025-04-16 11:22:05     86
2  552-ws-training 2025-04-16 11:27:05     81
3  552-ws-training 2025-04-16 11:32:05     81
4  552-ws-training 2025-04-16 11:37:05     82


In [34]:
# DATA VALIDATION FOR OhioT1DM DATASET

def validate_ohiot1dm(data):
    """
    Comprehensive data validation for OhioT1DM datasets.
    
    Checks:
    - Missing values
    - Data types
    - Value ranges
    - Duplicates
    - Timestamp consistency
    - Patient coverage
    """
    
    print("="*70)
    print("OhioT1DM DATA VALIDATION REPORT")
    print("="*70)
    
    # Overall statistics
    print("\nDATASET OVERVIEW")
    print("-" * 70)
    for event_type, df in data.items():
        if not df.empty:
            print(f"{event_type:12} | Rows: {len(df):>10,} | Patients: {df['patient_id'].nunique():>3} | "
                  f"Date Range: {df['ts'].min().date()} to {df['ts'].max().date()}")
        else:
            print(f"{event_type:12} | No data")
    
    # Missing values
    print("\n\nMISSING VALUES")
    print("-" * 70)
    has_missing = False
    for event_type, df in data.items():
        if not df.empty:
            missing = df.isnull().sum()
            if missing.sum() > 0:
                has_missing = True
                missing_pct = (missing / len(df) * 100).round(2)
                print(f"\n{event_type.upper()}:")
                for col in missing[missing > 0].index:
                    print(f"  {col:20} | Missing: {missing[col]:>6} ({missing_pct[col]:>5.1f}%)")
    
    if not has_missing:
        print("No missing values detected!")
    
    # Data types
    print("\n\nDATA TYPES")
    print("-" * 70)
    for event_type, df in data.items():
        if not df.empty:
            print(f"\n{event_type.upper()}:")
            print(df.dtypes)
    
    # Value ranges and outliers
    print("\n\n VALUE RANGES & OUTLIERS")
    print("-" * 70)
    
    # CGM (blood glucose) - normal range 70-180 mg/dL
    if not data['cgm'].empty:
        print("\nCGM (Blood Glucose):")
        cgm_vals = data['cgm']['value'].dropna()
        print(f"  Range: {cgm_vals.min():.1f} - {cgm_vals.max():.1f} mg/dL")
        print(f"  Mean:  {cgm_vals.mean():.1f} mg/dL")
        print(f"  Median: {cgm_vals.median():.1f} mg/dL")
        
        # Check for implausible values
        low = (cgm_vals < 20).sum()
        high = (cgm_vals > 600).sum()
        if low > 0 or high > 0:
            print(f"  IMPLAUSIBLE: {low} readings < 20 mg/dL, {high} readings > 600 mg/dL")
        else:
            print(f"  ✓ All values in plausible range")
    
    # Finger sticks
    if not data['finger'].empty:
        print("\nFinger Stick (Reference Glucose):")
        finger_vals = data['finger']['value'].dropna()
        print(f"  Range: {finger_vals.min():.1f} - {finger_vals.max():.1f} mg/dL")
        print(f"  Mean:  {finger_vals.mean():.1f} mg/dL")
        print(f"  Count: {len(finger_vals):,} measurements")
    
    # Bolus (insulin doses)
    if not data['bolus'].empty:
        print("\nBolus (Insulin Doses):")
        bolus_vals = data['bolus']['dose'].dropna()
        if len(bolus_vals) > 0:
            print(f"  Range: {bolus_vals.min():.2f} - {bolus_vals.max():.2f} units")
            print(f"  Mean:  {bolus_vals.mean():.2f} units")
            print(f"  Count: {len(bolus_vals):,} doses")
    
    # Meal carbs
    if not data['meal'].empty:
        print("\nMeal (Carbohydrates):")
        meal_vals = data['meal']['carbs'].dropna()
        if len(meal_vals) > 0:
            print(f"  Range: {meal_vals.min():.1f} - {meal_vals.max():.1f} grams")
            print(f"  Mean:  {meal_vals.mean():.1f} grams")
            print(f"  Count: {len(meal_vals):,} meals")
    
    # Duplicates
    print("\n\nDUPLICATE RECORDS")
    print("-" * 70)
    for event_type, df in data.items():
        if not df.empty:
            # Check for exact duplicates
            dups = df.duplicated().sum()
            # Check for same timestamp + patient
            ts_patient_dups = df[['patient_id', 'ts']].duplicated().sum()
            
            if dups > 0 or ts_patient_dups > 0:
                print(f"{event_type:12} | Exact duplicates: {dups:>6} | "
                      f"Same timestamp+patient: {ts_patient_dups:>6}")
            else:
                print(f"{event_type:12} | No duplicates")
    
    # Timestamp validation
    print("\n\nTIMESTAMP VALIDATION")
    print("-" * 70)
    for event_type, df in data.items():
        if not df.empty:
            # Check for any NaT (missing timestamps)
            nat_count = df['ts'].isna().sum()
            # Check if sorted
            is_sorted = df['ts'].is_monotonic_increasing
            # Time span
            time_span = (df['ts'].max() - df['ts'].min()).days
            
            print(f"\n{event_type.upper()}:")
            print(f"  First:  {df['ts'].min()}")
            print(f"  Last:   {df['ts'].max()}")
            print(f"  Span:   {time_span} days")
            print(f"  NaT:    {nat_count} missing timestamps")
            print(f"  Sorted: {'Yes' if is_sorted else 'No'}")
    
    # Patient coverage
    print("\n\nPATIENT COVERAGE")
    print("-" * 70)
    
    # Get all patients
    all_patients = set()
    for df in data.values():
        if not df.empty:
            all_patients.update(df['patient_id'].unique())
    
    print(f"Total unique patients: {len(all_patients)}")
    print(f"Patients: {sorted(all_patients)}\n")
    
    # Coverage matrix
    print("Data availability by patient:")
    print(f"{'Patient':<15}", end='')
    for event_type in data.keys():
        print(f"{event_type:<12}", end='')
    print()
    print("-" * 140)
    
    for patient in sorted(all_patients):
        print(f"{patient:<15}", end='')
        for event_type, df in data.items():
            if not df.empty:
                count = len(df[df['patient_id'] == patient])
                status = f"{count:>11}" if count > 0 else "EMPTY"
                print(f"{status:<12}", end='')
            else:
                print(f"{'N/A':<12}", end='')
        print()
    
    print("\n" + "="*70)

# Run validation
validate_ohiot1dm(data)

OhioT1DM DATA VALIDATION REPORT

DATASET OVERVIEW
----------------------------------------------------------------------
cgm          | Rows:     65,535 | Patients:   6 | Date Range: 2025-04-16 to 2027-07-03
finger       | Rows:      1,691 | Patients:   6 | Date Range: 2025-04-16 to 2027-07-03
basal        | Rows:        357 | Patients:   6 | Date Range: 2025-04-16 to 2027-06-22
bolus        | Rows:      1,568 | Patients:   6 | Date Range: NaT to NaT
meal         | Rows:        702 | Patients:   6 | Date Range: 2025-04-16 to 2027-07-03
sleep        | Rows:        150 | Patients:   5 | Date Range: NaT to NaT
exercise     | Rows:         61 | Patients:   4 | Date Range: 2025-04-17 to 2027-07-03
heartrate    | No data
steps        | No data


MISSING VALUES
----------------------------------------------------------------------

BASAL:
  duration             | Missing:    357 (100.0%)

BOLUS:
  ts                   | Missing:   1568 (100.0%)
  bwz_carb_input       | Missing:   1568 (100.0%

### 1.3 Data Validation

In [25]:
# TODO: Validate the dataset
# Check: how many '?' and '\t?' values exist? What columns have them?
# Also check: age range, classification distribution

print("="*60)
print("OhioT1DM CGM VALIDATION")
print("="*60)



OhioT1DM CGM VALIDATION


---
# PART 2: Schema Harmonization
---

Before linking, we need both datasets to speak the **same language**. This is the hardest part — and the most realistic.

### 2.1 Handle Missing Values

The UCI dataset encodes missing values as `'?'`, `'\t?'`, and sometimes `' ?'` (with leading spaces/tabs). Replace all of these with `NaN`.

In [26]:
# TODO: Replace all '?' variants with NaN in the UCI dataset
# Then convert numeric columns to their proper types
# Hint: uci = uci.replace(['?', '\t?', ' ?', '\t'], np.nan)
#        Then use pd.to_numeric() on columns that should be numeric


cgm = cgm.set_index('ts')
full_index = pd.date_range(cgm.index.min(), cgm.index.max(), freq='5min')
cgm = cgm.reindex(full_index)
cgm['value'] = cgm['value'].interpolate(method='time', limit=6)  # fill gaps up to 30 min
cgm = cgm.rename_axis('ts').reset_index()





'''
uci_clean = uci.copy()
                                                                            
# Step 1: Replace ? with NaN
uci_clean = uci_clean.replace(['?', '\t?', ' ?', '\t'], np.nan)

# Step 2: Convert numeric columns to float
numeric_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod',
              'pot', 'hemo', 'pcv', 'wc', 'rc']
for col in numeric_cols:
  uci_clean[col] = pd.to_numeric(uci_clean[col], errors='coerce')

print("Missing values per column:")
print(uci_clean.isnull().sum())
print(f"\nTotal missing: {uci_clean.isnull().sum().sum()}")
'''

ValueError: cannot reindex on an axis with duplicate labels

### 2.2 Rename UCI Columns to Readable Names

Replace the cryptic abbreviations with descriptive names that match the Clinical dataset.

In [ ]:
# TODO: Rename UCI CKD columns using the mapping below
# Hint: uci_clean = uci_clean.rename(columns=rename_map)

rename_map = {
    'age': 'age',
    'bp': 'diastolic_bp',
    'sg': 'specific_gravity',
    'al': 'albumin_ordinal',
    'su': 'sugar_ordinal',
    'rbc': 'rbc_status',
    'pc': 'pus_cell',
    'pcc': 'pus_cell_clumps',
    'ba': 'bacteria',
    'bgr': 'blood_glucose',
    'bu': 'blood_urea',
    'sc': 'serum_creatinine',
    'sod': 'sodium',
    'pot': 'potassium',
    'hemo': 'hemoglobin',
    'pcv': 'packed_cell_volume',
    'wc': 'wbc_count',
    'rc': 'rbc_count',
    'htn': 'hypertension',
    'dm': 'diabetes',
    'cad': 'coronary_artery_disease',
    'appet': 'appetite',
    'pe': 'pedal_edema',
    'ane': 'anemia',
    'classification': 'ckd_diagnosis'
}

uci_clean = uci_clean.rename(columns=rename_map)

print("Renamed UCI columns:")
print(uci_clean.columns.tolist())

### 2.3 Encode UCI String Categoricals to Binary

UCI uses "yes"/"no", "normal"/"abnormal", "present"/"notpresent" strings. Convert to 0/1.

In [ ]:
# TODO: Convert string categorical columns to binary 0/1
# Mappings:
#   "yes" -> 1, "no" -> 0  (for hypertension, diabetes, cad, pedal_edema, anemia)
#   "good" -> 1, "poor" -> 0  (for appetite)
#   "normal" -> 0, "abnormal" -> 1  (for rbc_status, pus_cell)
#   "present" -> 1, "notpresent" -> 0  (for pus_cell_clumps, bacteria)
#   "ckd" -> 1, "notckd" -> 0  (for ckd_diagnosis)
#
# Hint: Handle extra whitespace! Some values have leading/trailing spaces or tabs.
#        Use .str.strip() first: uci_clean['col'] = uci_clean['col'].str.strip()

# Step 1: Strip whitespace from ALL string columns
str_cols = uci_clean.select_dtypes(include='object').columns
for col in str_cols:
    uci_clean[col] = uci_clean[col].str.strip()

# Step 2: Apply Mappings
mappings = {
  'yes_no': {'yes': 1, 'no': 0},
  'good_poor': {'good': 1, 'poor': 0},
  'normal_abnormal': {'normal': 0, 'abnormal': 1},
  'present_notpresent': {'present': 1, 'notpresent': 0},
  'ckd_notckd': {'ckd': 1, 'notckd': 0}
}

for col in ['hypertension', 'diabetes', 'coronary_artery_disease', 'pedal_edema', 'anemia']:
  uci_clean[col] = uci_clean[col].map(mappings['yes_no'])

uci_clean['appetite'] = uci_clean['appetite'].map(mappings['good_poor'])

for col in ['rbc_status', 'pus_cell']:
  uci_clean[col] = uci_clean[col].map(mappings['normal_abnormal'])

for col in ['pus_cell_clumps', 'bacteria']:
  uci_clean[col] = uci_clean[col].map(mappings['present_notpresent'])

uci_clean['ckd_diagnosis'] = uci_clean['ckd_diagnosis'].map(mappings['ckd_notckd'])

# Verify
print("UCI binary columns after encoding:")
for col in ['hypertension', 'diabetes', 'coronary_artery_disease', 'pedal_edema', 'anemia', 'appetite', 'rbc_status', 'pus_cell',
'ckd_diagnosis']:
  print(f"  {col}: {uci_clean[col].value_counts(dropna=False).to_dict()}")

### 2.4 Standardize CKD Clinical Column Names

The Clinical dataset uses CamelCase. Standardize to snake_case for consistency.

In [ ]:
# TODO: Convert Clinical column names to snake_case
# Hint: Use a regex or manual approach
# Simple approach: clinical.columns = clinical.columns.str.replace(r'(?<=[a-z])(?=[A-Z])', '_', regex=True).str.lower()

clinical.columns = clinical.columns.str.replace(r'(?<=[a-z])(?=[A-Z])', '_', regex=True).str.lower()

print("Standardized Clinical columns:")
print(clinical.columns.tolist())

### 2.5 Create Age Bins in Both Datasets

Create matching age groups for groupby-based linking.

In [ ]:
# TODO: Create 'age_bin' in both datasets using pd.cut
# Bins: [0, 29, 44, 59, 74, 100]
# Labels: ['Under 30', '30-44', '45-59', '60-74', '75+']

bins = [0, 29, 44, 59, 74, 100]
labels = ['Under 30', '30-44', '45-59', '60-74', '75+']

uci_clean['age_bin'] = pd.cut(uci_clean['age'], bins=bins, labels=labels)
clinical['age_bin'] = pd.cut(clinical['age'], bins=bins, labels=labels)

print("UCI Age Bins:")
print(uci_clean['age_bin'].value_counts().sort_index())
print("\nClinical Age Bins:")
print(clinical['age_bin'].value_counts().sort_index())

### 2.6 Create Creatinine Risk Categories

Serum creatinine is shared by both datasets. Create clinical risk categories for merging.

> 💡 **Clinical Context:**
> - Normal creatinine: 0.6–1.2 mg/dL
> - Mildly elevated: 1.3–2.0 mg/dL
> - Moderately elevated: 2.1–5.0 mg/dL
> - Severely elevated: > 5.0 mg/dL

In [ ]:
uci_clean.columns

In [ ]:
# TODO: Create 'creatinine_cat' in both datasets using pd.cut
# Bins: [0, 1.2, 2.0, 5.0, 50]
# Labels: ['Normal', 'Mild', 'Moderate', 'Severe']
# Note: The Clinical column may be named 'serum_creatinine' after snake_case conversion

creat_bins = [0, 1.2, 2.0, 5.0, 50]
creat_labels = ['Normal', 'Mild', 'Moderate', 'Severe']

uci_clean['creatinine_cat'] = pd.cut(uci_clean['serum_creatinine'], bins=creat_bins, labels=creat_labels)

# Find the creatinine column name in the clinical dataset
# Hint: It might be 'serum_creatinine' or 'serumcreatinine' after snake_case conversion
# Use clinical.columns to check!

clinical['creatinine_cat'] = pd.cut(clinical['serum_creatinine'], bins=creat_bins, labels=creat_labels)


print("UCI Creatinine Categories:")
print(uci_clean['creatinine_cat'].value_counts().sort_index())
print("\nClinical Creatinine Categories:")
print(clinical['creatinine_cat'].value_counts().sort_index())

---
# PART 3: Strategy 1 — Vertical Stack (`pd.concat`) - Tyler
---

**Goal:** Align shared columns, stack rows from both datasets into one DataFrame.

### 📋 Shared columns to align:
- `age` — both datasets (years)
- `serum_creatinine` — both datasets (mg/dL)
- `hemoglobin` — UCI: `hemoglobin`, Clinical: check column name after conversion
- `blood_urea` — UCI: `blood_urea`, Clinical: check BUN column name
- `sodium` — UCI: `sodium`, Clinical: check column name
- `potassium` — UCI: `potassium`, Clinical: check column name
- `target` — UCI: `ckd_diagnosis` (0/1), Clinical: `diagnosis` (0/1)

In [ ]:
# TODO: Create uci_aligned DataFrame with columns:
# age, serum_creatinine, hemoglobin, blood_urea, sodium, potassium,
# age_bin, creatinine_cat, target (from ckd_diagnosis), dataset_source='uci'
#
# Hint: You may need to check exact column names after your renaming/standardization
#
# 1. Selects the 8 shared columns from uci_clean
# 2. Adds target column (mapped from ckd_diagnosis)
# 3. Adds dataset_source column with value 'uci'
# 4. Displays the shape and first 3 rows

uci_aligned = uci_clean[['age', 'serum_creatinine', 'hemoglobin', 'blood_urea', 'sodium', 'potassium', 'age_bin', 'creatinine_cat']].copy()
uci_aligned['target'] = uci_clean['ckd_diagnosis']
uci_aligned['dataset_source'] = 'uci'

# Display shape and first 3 rows
print("UCI aligned:", uci_aligned.shape)
uci_aligned.head(3)

In [ ]:
# TODO: Create clinical_aligned DataFrame with matching columns
# Map clinical column names to match uci_aligned columns
# age, serum_creatinine, hemoglobin, blood_urea, sodium, potassium,
# age_bin, creatinine_cat, target (from diagnosis column), dataset_source='clinical'
#
# Hint: Check clinical.columns to find exact names for hemoglobin, BUN, sodium, potassium
#
# 1. Auto-detects the exact column names for hemoglobin, BUN, sodium, and potassium by searching for keywords (case-insensitive search)
# 2. Selects the 8 shared columns from clinical
# 3. Renames them to match the uci_aligned schema
# 4. Adds target column (mapped from diagnosis)
# 5. Adds dataset_source column with value 'clinical'
# 6. Displays the shape and first 3 rows

hemoglobin_col = next(col for col in clinical.columns if 'hemoglobin' in col.lower())
bun_col = next(col for col in clinical.columns if 'bun' in col.lower())
sodium_col = next(col for col in clinical.columns if 'sodium' in col.lower())
potassium_col = next(col for col in clinical.columns if 'potassium' in col.lower())

clinical_aligned = clinical[['age', 'serum_creatinine', hemoglobin_col, bun_col, sodium_col, potassium_col, 'age_bin', 'creatinine_cat']].copy()
clinical_aligned.columns = ['age', 'serum_creatinine', 'hemoglobin', 'blood_urea', 'sodium', 'potassium', 'age_bin', 'creatinine_cat']
clinical_aligned['target'] = clinical['diagnosis']
clinical_aligned['dataset_source'] = 'clinical'

print("Clinical aligned:", clinical_aligned.shape)
clinical_aligned.head(3)

In [ ]:
# TODO: Stack both DataFrames using pd.concat
# Use ignore_index=True, then drop NaN rows with .dropna()

# Stack uci_aligned and clinical_aligned vertically using pd.concat()
# ignore_index=True creates a new sequential index (0, 1, 2, ...)
# .dropna() removes any rows with missing values
combined = pd.concat([uci_aligned, clinical_aligned], ignore_index=True)
combined = combined.dropna()

# Display the shape and first 3 rows
print(f"Combined shape: {combined.shape[0]:,} rows")
print(f"\nBy dataset_source:")
print(combined['dataset_source'].value_counts())
combined.head(3)

In [ ]:
# TODO: Create overlaid histograms of serum_creatinine by dataset_source
# Use density=True for fair comparison (different sample sizes!)
# Use different colors and alpha=0.5
# Limit x-axis to [0, 15] to handle outliers

fig, ax = plt.subplots(figsize=(10, 5))

# Separates UCI and Clinical data by filtering dataset_source

uci_data = combined[combined['dataset_source'] == 'uci']['serum_creatinine']
clinical_data = combined[combined['dataset_source'] == 'clinical']['serum_creatinine']

# Creates overlaid histograms for each dataset source with density=True (for fair comparison despite different sample sizes)
# Uses alpha=0.5 for transparency so both histograms are visible
# Uses blue for UCI, orange for Clinical in plot
# 30 bins for good granularity

ax.hist(uci_data, bins=30, density=True, alpha=0.5, label='UCI CKD', color='blue')
ax.hist(clinical_data, bins=30, density=True, alpha=0.5, label='Clinical', color='orange')

# Limit x-axis to [0, 15] to handle outliers

ax.set_xlabel('Serum Creatinine (mg/dL)')
ax.set_ylabel('Density')
ax.set_title('Serum Creatinine Distribution by Source')
ax.set_xlim(0, 15)
ax.legend()
plt.show()

ax.set_xlabel('Serum Creatinine (mg/dL)')
ax.set_ylabel('Density')
ax.set_title('Serum Creatinine Distribution by Source')
ax.set_xlim(0, 15)
ax.legend()
plt.show()

**Question:** How do the serum creatinine distributions differ between the UCI and Clinical datasets? What does this tell us about the patient populations in each study?

*Your answer:* 
The UCI CKD dataset is strongly right-skewed with a massive spike at 0–1 mg/dL, followed by rapidly declining density. It also has a very long tail extending all the way to 15+ mg/dL, with scattered low-density bars from 5–15 mg/dL. Most of the patients in this dataset are clustered below 2 mg/dL.

The Clinical dataset is much flatter and has a more uniform distribution spread across 0–5 mg/dL. The density stays relatively consistent (~0.20–0.28) across a wide range with no no single dominant spike. It then drops off sharply at ~5 mg/dL with virtually no extreme values after that.

---
## PART 4: Strategy 2 — Aggregate Enrichment (`groupby` + `merge`) - Kem
---

**Goal:** Build a clinical-profile lookup table from the richer Clinical dataset, then merge it into the UCI dataset to enrich each UCI patient with population-level GFR, medication usage, lipid levels, and quality-of-life scores.

> 💡 **Why this works:** The UCI dataset is a bare-bones lab panel — it has creatinine and urea but no GFR, no medication history, no lipid panel, no QoL scores. The Clinical dataset has all of this. By grouping Clinical patients by `[age_bin, creatinine_cat]` and computing averages, we can enrich UCI patients with the **typical clinical profile** for their demographic and kidney-function stratum.

### 4.1 Build the Clinical Profile Lookup Table

In [ ]:
clinical.columns

In [ ]:
# TODO: Group the clinical dataset by ['age_bin', 'creatinine_cat'] and compute:
# - avg_gfr: mean of GFR column (find exact name after snake_case conversion)
# - avg_hba1c: mean of HbA1c column
# - avg_cholesterol: mean of total cholesterol column
# - avg_bmi: mean of BMI column
# - ace_inhibitor_rate: mean of ACE inhibitor column
# - nsaid_rate: mean of NSAID usage column
# - avg_qol: mean of quality of life score column
# - ckd_rate: mean of diagnosis column (proportion with CKD)
# - n_patients: count of diagnosis column
# Then .round(3).reset_index()
#
# Hint: Check clinical.columns to find exact column names!
# The snake_case conversion may produce different results depending on your approach.

# First, find the right column names:
print("Clinical columns (for reference):")
print(clinical.columns.tolist())

lookup = clinical.groupby(['age_bin', 'creatinine_cat']).agg(                                              
  avg_gfr=('gfr', 'mean'),
  avg_hba1c=('hb_a1c', 'mean'),                                                                          
  avg_cholesterol=('cholesterol_total', 'mean'),
  avg_bmi=('bmi', 'mean'),
  ace_inhibitor_rate=('aceinhibitors', 'mean'),
  nsaid_rate=('nsaids_use', 'mean'),
  avg_qol=('quality_of_life_score', 'mean'),
  ckd_rate=('diagnosis', 'mean'),
  n_patients=('diagnosis', 'count')
).round(3).reset_index()


print(f"\nLookup table: {len(lookup)} groups")
display(lookup)

### 4.2 Merge Clinical Profiles into UCI CKD

In [ ]:
# TODO: Left merge the lookup table into uci_clean on ['age_bin', 'creatinine_cat']

uci_enriched = uci_clean.merge(lookup, on=['age_bin', 'creatinine_cat'], how='left')
    
print(f"Enriched shape: {uci_enriched.shape}")
print(f"NaN in avg_gfr: {uci_enriched['avg_gfr'].isnull().sum()} out of {len(uci_enriched)}")

In [ ]:
# TODO: Compare enriched features between CKD (1) and non-CKD (0)
# Group uci_enriched by 'ckd_diagnosis' and show the mean of:
# avg_gfr, avg_hba1c, ace_inhibitor_rate, nsaid_rate, avg_qol

enriched_cols = ['avg_gfr', 'avg_hba1c', 'ace_inhibitor_rate', 'nsaid_rate', 'avg_qol']

# YOUR CODE HERE

comparison = uci_enriched.groupby('ckd_diagnosis')[enriched_cols].mean()                                                          
display(comparison)      

In [ ]:
# TODO: Create a visualization comparing enriched profiles by CKD status
# Suggestion: horizontal bar chart or grouped bar chart

fig, ax = plt.subplots(figsize=(10, 5))

# YOUR CODE HERE
enriched_cols = ['avg_gfr', 'avg_hba1c', 'ace_inhibitor_rate', 'nsaid_rate', 'avg_qol']                                                                 
comparison = uci_enriched.groupby('ckd_diagnosis')[enriched_cols].mean()    
                                                                              
# Transpose to get features as rows, CKD status as columns                  
comparison.T.plot(kind='barh', ax=ax)                                       
ax.set_xlabel('Mean Value')                                                 
ax.set_ylabel('Clinical Features')
ax.legend(['Non-CKD (0)', 'CKD (1)'], title='Diagnosis')
                    
ax.set_title('Population-Level Clinical Profiles by CKD Diagnosis')

plt.tight_layout()
plt.show()

In [ ]:
print("Comparison values:")
print(comparison)
print("\n")

**Question:** Do CKD patients in the UCI dataset fall into age/creatinine groups with lower average GFR and higher medication usage rates? What does enrichment with GFR add that the original creatinine measurement alone does not tell us?

*Your answer:*
The difference of GFR and medication usage rates is actually flipped. Non-CKD Patients seem to be showing higher average GFR and lower medication usage rates.

Non-CKD vs CKD  
GFR (66.80 vs 65.91)  
NSAID Rate (5.08 vs 4.96)  

The GFR enrichment adds more information to the original creatinine measurement.  
It gives us data on what the average for each category is; in this case the categories we're comparing are CKD patients and non-CKD patients.  
It makes it easier to compare what non-CKD patient lab results are like compared to the patients with CKD usually get for their values.  

---
# PART 5: Strategy 3 — Train-Transfer (Model as Bridge)
---

**Goal:** Train a LogisticRegression on the Clinical dataset to predict CKD diagnosis. Then use this model to generate a "CKD risk score" for each UCI patient.

> 💡 **The idea:** The Clinical dataset has features (GFR, HbA1c, BMI, cholesterol) that are strong predictors of CKD. UCI patients don't have these directly, but we enriched them in Strategy 2. We'll now **chain Strategy 2 into Strategy 3** — using the enriched population-level values as proxy inputs for the transfer model.

**Approach:**
1. Train on Clinical: **age + creatinine + GFR + HbA1c + BMI → CKD diagnosis**
2. For UCI patients: use their actual age and creatinine, plus enriched GFR, HbA1c, BMI from Strategy 2
3. Generate CKD risk scores using `predict_proba`

### 5.1 Prepare Training Data

In [ ]:
clinical.columns

In [ ]:
# TODO: Create X_clinical with columns from the clinical dataset:
# ['age', 'serum_creatinine_col', 'gfr_col', 'hba1c_col', 'bmi_col']
# (replace with actual column names from your clinical dataset)
# Create y_clinical from the diagnosis column
# Drop any rows with NaN
#
# Hint: Check clinical.columns for exact names!

train_features_clinical = ['age', 'serum_creatinine', 'gfr', 'hb_a1c', 'bmi']  
display_names = ['age', 'serum_creatinine', 'gfr', 'hba1c', 'bmi']

X_clinical = clinical[train_features_clinical]
y_clinical = clinical['diagnosis']

# Drop NaN
mask = X_clinical.notna().all(axis=1) & y_clinical.notna()
X_clinical = X_clinical[mask]
y_clinical = y_clinical[mask]

print(f"Training data: {X_clinical.shape[0]:,} rows, {X_clinical.shape[1]} features")
print(f"CKD rate: {y_clinical.mean()*100:.1f}%")

### 5.2 Train the Model

In [ ]:
# TODO: Scale features with StandardScaler, then train LogisticRegression
# 1. Create scaler, fit_transform X_clinical
# 2. Create model with random_state=42, max_iter=1000
# 3. Fit model on scaled data

scaler = StandardScaler() 
X_clinical_scaled = scaler.fit_transform(X_clinical)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_clinical_scaled, y_clinical)

print(f"Training accuracy: {model.score(X_clinical_scaled, y_clinical)*100:.1f}%")

### 5.3 Generate CKD Risk Scores for UCI Patients

Since UCI patients don't have GFR, HbA1c, or BMI individually, we use their **enriched values** from Strategy 2 (population-level averages based on age bin and creatinine category).

In [ ]:
# TODO: Prepare UCI enriched data for prediction
# Create X_uci DataFrame with columns matching the training features:
#   'age' from uci_enriched['age']
#   'serum_creatinine' from uci_enriched['serum_creatinine']
#   'gfr' from uci_enriched['avg_gfr']
#   'hba1c' from uci_enriched['avg_hba1c']
#   'bmi' from uci_enriched['avg_bmi']  (if available, else avg_bmi from lookup)
#
# Drop NaN, scale using SAME scaler (transform, NOT fit_transform!), predict_proba

X_uci = uci_enriched[['age', 'serum_creatinine', 'avg_gfr', 'avg_hba1c', 'avg_bmi']].copy()
X_uci.columns = ['age', 'serum_creatinine', 'gfr', 'hb_a1c', 'bmi']

# Drop NaN
uci_for_transfer = uci_enriched.copy()
mask = X_uci.notna().all(axis=1)
X_uci = X_uci[mask]
uci_for_transfer = uci_for_transfer[mask]

# Scale and predict
X_uci_scaled = scaler.transform(X_uci)
ckd_risk_scores = model.predict_proba(X_uci_scaled)[:, 1]

uci_for_transfer['ckd_risk_score'] = ckd_risk_scores

print(f"Risk score range: {ckd_risk_scores.min():.3f} to {ckd_risk_scores.max():.3f}")
print(f"Mean risk score: {ckd_risk_scores.mean():.3f}")

### 5.4 Analyze Results

In [ ]:
# TODO: Compare CKD risk scores between ckd_diagnosis=1 (CKD) and ckd_diagnosis=0 (not CKD)
# Use groupby and describe()

comparison = uci_for_transfer.groupby('ckd_diagnosis')['ckd_risk_score'].describe()                        
print(comparison)

In [ ]:
# TODO: Create a box plot of ckd_risk_score by ckd_diagnosis

plt.figure(figsize=(8, 6))                                                                                 
sns.boxplot(data=uci_for_transfer, x='ckd_diagnosis', y='ckd_risk_score')
plt.xlabel('CKD Diagnosis (0=No, 1=Yes)')                                                                  
plt.ylabel('CKD Risk Score')
plt.title('CKD Risk Score Distribution by Diagnosis')
plt.show()

In [ ]:
# TODO: Create a scatter plot of serum_creatinine vs ckd_risk_score
# Color by ckd_diagnosis
# This shows how the original lab value relates to the cross-dataset risk score

fig, ax = plt.subplots(figsize=(10, 6))                                                                    

sns.scatterplot(data=uci_for_transfer, x='serum_creatinine', y='ckd_risk_score', hue='ckd_diagnosis',      
ax=ax, alpha=0.6)

ax.set_xlabel('Serum Creatinine (mg/dL)')
ax.set_ylabel('CKD Risk Score (from Clinical model)')
ax.set_title('Lab Value vs. Cross-Dataset Risk Score')
ax.set_xlim(0, 15)
ax.legend(title='CKD Diagnosis', labels=['No CKD (0)', 'CKD (1)'])
plt.tight_layout()
plt.show()

**Question:** Does the CKD risk score from the Clinical model successfully separate CKD from non-CKD patients in the UCI dataset? What does the scatter plot reveal about the relationship between creatinine and the risk score? Why might using population-level GFR/HbA1c averages (rather than individual measurements) limit the model's discriminative power?

**Answer:**  
The CKD risk score does not cleanly separate CKD from non-CKD patients in the UCI dataset. The orange and blue dots are mixed together; you can't draw a clear line between them (orange = No CKD, blue = CKD) overlap significantly — especially in the 0.5–2.0 mg/dL creatinine range, where both groups cluster together with similar risk scores.  

This is partly because instead of using each patient's real GFR and HbA1c values, we used group averages (e.g., "all 50-year-olds with high creatinine get the same average GFR"). So patients who are actually very different end up looking the same to the model.  

Basically, the model is mostly just reacting to creatinine levels rather than truly using multiple features — which is why the plot looks like a smooth curve instead of two separated clusters.  

In short, the merge strategy adds features but those features are too coarsely grouped to add meaningful discriminative information beyond what creatinine already provides.

---
# PART 6: Challenge Exercises
---

### 🚀 Challenge 1: Comorbidity Burden Score

Create a "comorbidity burden" score for UCI patients by summing their binary comorbidity columns (hypertension, diabetes, coronary_artery_disease, anemia). Then compare the mean CKD risk score across different comorbidity burden levels (0, 1, 2, 3+). Does higher comorbidity burden correlate with higher CKD risk from the Clinical model?

In [ ]:
# CHALLENGE 1: Comorbidity burden analysis

# Create comorbidity burden score by summing binary comorbidity columns
comorbidity_cols = ['hypertension', 'diabetes', 'coronary_artery_disease', 'anemia']
uci_for_transfer['comorbidity_burden'] = uci_for_transfer[comorbidity_cols].sum(axis=1)

# Create burden categories (0, 1, 2, 3+)
uci_for_transfer['burden_category'] = pd.cut(uci_for_transfer['comorbidity_burden'], bins=[-0.5, 0.5, 1.5, 2.5,
  5], labels=['0', '1', '2', '3+'])

# Compare mean CKD risk score across burden levels
print("Mean CKD Risk Score by Comorbidity Burden:")
print(uci_for_transfer.groupby('burden_category', observed=True)['ckd_risk_score'].agg(['mean', 'std', 'count']))

# Create visualization for UCI
fig, ax = plt.subplots(figsize=(10, 6))
uci_for_transfer.groupby('burden_category', observed=True)['ckd_risk_score'].mean().plot(kind='bar', ax=ax,
color='steelblue')
ax.set_xlabel('Comorbidity Burden Level')
ax.set_ylabel('Mean CKD Risk Score')
ax.set_title('CKD Risk Score by Comorbidity Burden')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()



- Even patients with zero comorbidities have a high mean score. This is due to our data coming from a clinical population that is already likely to be of higher average risk.

- Overall, the data shows a clear positive correlation between comorbidity burden and CKD risk score.

- The 3+ group has the smallest standard deviation (0.032), meaning high-burden patients are consistently high-risk, not just on average

- However, the sample is heavily imbalanced — 187 patients have no comorbidities vs. only 26 in the 3+ group. Therefore, the 3+ mean should be interpreted with some caution.

### 🚀 Challenge 2: GFR Stage Enrichment

Using the Clinical dataset, create CKD stage categories based on GFR:
- Stage 1: GFR ≥ 90
- Stage 2: GFR 60–89
- Stage 3: GFR 30–59
- Stage 4: GFR 15–29
- Stage 5: GFR < 15

Build a lookup table grouped by `[age_bin, ckd_stage]` with medication rates and QoL scores. Merge into UCI patients using their enriched GFR values.

In [ ]:
# CHALLENGE 2: GFR-based staging and enrichment

# YOUR CODE HERE

### 🚀 Challenge 3: Feature Importance

Using the trained LogisticRegression model, create a bar chart of feature coefficients. Which clinical features most strongly predict CKD? How does this align with clinical knowledge about kidney disease?

In [ ]:
# CHALLENGE 3: Feature importance bar chart
# Hint: model.coef_[0] gives the coefficients
# Use display_names for labels

# YOUR CODE HERE

# Get the coefficients from the trained model
coefficients = model.coef_[0]

# Plot the coefficients as a bar chart
plt.figure(figsize=(8, 5))
plt.bar(display_names, coefficients, color='steelblue')
plt.axhline(y=0, color='black', linewidth=0.8)
plt.xlabel('Feature')
plt.ylabel('Coefficient')
plt.title('Which Features Most Strongly Predict CKD?')
plt.tight_layout()
plt.show()

# Print the values too
for name, coef in zip(display_names, coefficients):
    print(f"{name}: {coef:.3f}")

- Serum creatinine and GFR most strongly predict CKD. Serum creatinine had the highest positive coefficient (0.870) and GFR had the highest negative coefficient (-0.739).
- This aligns with clinical knowledge because CKD is directly defined by reduced kidney filtration. When kidneys fail, creatinine builds up in the blood (high creatinine) and filtration drops (low GFR), so these two features are exactly what doctors use to diagnose CKD.

### 🚀 Challenge 4: Full Enrichment Pipeline

Create a single UCI DataFrame that has:
1. Original UCI columns (decoded and cleaned)
2. Population-level clinical profiles from Strategy 2 (GFR, HbA1c, medication rates, QoL)
3. Individual-level CKD risk score from Strategy 3
4. The comorbidity burden score from Challenge 1

Then compute a correlation matrix of key features (`serum_creatinine`, `hemoglobin`, `blood_urea`, `ckd_risk_score`, `avg_gfr`, `avg_hba1c`) vs. `ckd_diagnosis`.

In [ ]:
# CHALLENGE 4: Full enrichment + correlation analysis

# The uci_for_transfer already has all the enriched columns
# Create correlation matrix of key features vs diagnosis
key_features = ['serum_creatinine', 'hemoglobin', 'blood_urea', 'ckd_risk_score', 'avg_gfr', 'avg_hba1c']

# Compute correlations
correlation_matrix = uci_for_transfer[key_features + ['ckd_diagnosis']].corr()

# Show correlation with diagnosis
print("Correlation with CKD Diagnosis:")
print(correlation_matrix['ckd_diagnosis'].drop('ckd_diagnosis').sort_values(ascending=False))

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix: Key Features vs CKD Diagnosis')
plt.tight_layout()
plt.show()

- Individual features matter: Serum_creatinine, GFR, and HbA1c all correlate with CKD diagnosis 
- But they overlap: When combined in the model, some features get low coefficients because they're
measuring similar things 
- Model still works: Despite this, the trained model successfully separates CKD from non-CKD patients in
the UCI dataset
- Risk scores show signal: The scatter plot reveals that higher serum_creatinine generally leads to higher
risk scores, confirming the model learned the right pattern
- Trade-off: Using population-level averages (avg_gfr, avg_hba1c) instead of individual measurements limits
the model's power to discriminate between patients



---
## ✅ Submission Checklist

Before submitting, make sure you have:

- [ ] Completed all TODO sections in Parts 1–5
- [ ] Run all cells to verify they work (Kernel → Restart & Run All)
- [ ] Answered all written questions
- [ ] Attempted at least 2 of the 4 challenge exercises
- [ ] Saved your notebook

---

## Quick Reference

| Task | Code |
|------|------|
| Load CSV | `pd.read_csv('file.csv')` |
| Replace values | `df.replace(['?', '\t?'], np.nan)` |
| Strip whitespace | `df['col'].str.strip()` |
| Rename columns | `df.rename(columns={'old': 'new'})` |
| CamelCase → snake_case | `df.columns.str.replace(r'(?<=[a-z])(?=[A-Z])', '_', regex=True).str.lower()` |
| Map values | `df['col'].map({'yes': 1, 'no': 0})` |
| pd.to_numeric | `pd.to_numeric(df['col'], errors='coerce')` |
| Concat rows | `pd.concat([df1, df2], ignore_index=True)` |
| GroupBy + Agg | `df.groupby(['col1','col2']).agg(name=('col', 'func'))` |
| Merge | `df1.merge(df2, on=['key1','key2'], how='left')` |
| pd.cut (binning) | `pd.cut(df['col'], bins=[...], labels=[...])` |
| Scale features | `StandardScaler().fit_transform(X)` |
| Train model | `LogisticRegression().fit(X, y)` |
| Predict proba | `model.predict_proba(X)[:, 1]` |

---

### Clinical Reference: CKD Stages by GFR

| Stage | GFR (mL/min) | Description |
|-------|-------------|-------------|
| 1 | ≥ 90 | Normal or high |
| 2 | 60–89 | Mildly decreased |
| 3a | 45–59 | Mildly to moderately decreased |
| 3b | 30–44 | Moderately to severely decreased |
| 4 | 15–29 | Severely decreased |
| 5 | < 15 | Kidney failure |

---

## Remember Our Mantra:

# "One Patient, Many Datasets"

---

**See you in class!** 🎓